In [27]:
from pathlib import Path
from awpy import Demo

# go from /src/analytics → project root
BASE_DIR = Path().resolve().parents[2] / "Internomat" 

DEMO_PATH = BASE_DIR / "test_demos" / "2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem"

print("Resolved path:", DEMO_PATH)
print("Exists:", DEMO_PATH.exists())


def load_demo(path: Path):
    path = path.resolve()

    if not path.exists():
        raise FileNotFoundError(f"Demo not found: {path}")

    print(f"[Demo] Loading: {path}")

    demo = Demo(path=str(path))
    demo.parse()

    print("[Demo] Parsing complete.")
    return demo


demo = load_demo(DEMO_PATH)

Resolved path: D:\Git_repo\Internomat\test_demos\2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem
Exists: True
[Demo] Loading: D:\Git_repo\Internomat\test_demos\2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem
[Demo] Parsing complete.


In [36]:
def inspect_demo_attributes(demo):
    print("\n=== AVAILABLE DEMO ATTRIBUTES ===")

    attrs = dir(demo)

    # filter useful ones
    public_attrs = [a for a in attrs if not a.startswith("_")]

    for attr in sorted(public_attrs):
        try:
            value = getattr(demo, attr)

            if isinstance(value, list):
                print(f"{attr:20} -> list ({len(value)} items)")
            elif isinstance(value, dict):
                print(f"{attr:20} -> dict ({len(value)} keys)")
            else:
                print(f"{attr:20} -> {type(value).__name__}")

        except Exception as e:
            print(f"{attr:20} -> ERROR: {e}")


inspect_demo_attributes(demo)


=== AVAILABLE DEMO ATTRIBUTES ===
bomb                 -> DataFrame
compress             -> method
damages              -> DataFrame
default_events       -> list (20 items)
detected_events      -> list (42 items)
events               -> dict (21 keys)
footsteps            -> DataFrame
grenades             -> DataFrame
header               -> dict (12 keys)
in_play_ticks        -> Series
inferno_duration     -> float
infernos             -> DataFrame
kills                -> DataFrame
parse                -> method
parse_events         -> method
parse_grenades       -> method
parse_header         -> method
parse_ticks          -> method
parser               -> DemoParser
path                 -> WindowsPath
player_round_totals  -> DataFrame
rounds               -> DataFrame
server_cvars         -> DataFrame
shots                -> DataFrame
smoke_duration       -> float
smokes               -> DataFrame
tickrate             -> int
ticks                -> DataFrame


In [41]:
import pandas as pd

def parse_demo_full(demo):
    print("\n[Demo] Building structured tables (explicit)...")

    data = {}

    # --- HEADER ---
    data["header"] = demo.header

    # --- KNOWN TABLES (from your inspection) ---
    table_map = {
        "rounds": "rounds",
        "kills": "kills",
        "damages": "damages",
        "grenades": "grenades",
        "shots": "shots",
        "footsteps": "footsteps",
        "smokes": "smokes",
        "infernos": "infernos",
        "bomb": "bomb",
        "ticks": "ticks",
        "player_round_totals": "player_round_totals",
        "server_cvars": "server_cvars",
    }

    for key, attr in table_map.items():
        try:
            value = getattr(demo, attr, None)

            if value is None:
                print(f"[NULL] {key}")
                data[key] = None
                continue

            # force into DataFrame if possible
            if isinstance(value, pd.DataFrame):
                df = value
            elif isinstance(value, list):
                df = pd.DataFrame(value)
            else:
                df = value  # keep as-is (e.g. ticks may already be structured)

            data[key] = df

            if isinstance(df, pd.DataFrame):
                print(f"[OK]   {key:22} -> {len(df)} rows")
            else:
                print(f"[OK]   {key:22} -> {type(df).__name__}")

        except Exception as e:
            print(f"[FAIL] {key}: {e}")
            data[key] = None

    print("\n[Demo] Total collected:", len(data))

    return data


data = parse_demo_full(demo)


[Demo] Building structured tables (explicit)...
[OK]   rounds                 -> DataFrame
[OK]   kills                  -> DataFrame
[OK]   damages                -> DataFrame
[OK]   grenades               -> DataFrame
[OK]   shots                  -> DataFrame
[OK]   footsteps              -> DataFrame
[OK]   smokes                 -> DataFrame
[OK]   infernos               -> DataFrame
[OK]   bomb                   -> DataFrame
[OK]   ticks                  -> DataFrame
[OK]   player_round_totals    -> DataFrame
[OK]   server_cvars           -> DataFrame

[Demo] Total collected: 13
